In [ ]:
import os
from datetime import datetime
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_tavily import TavilySearch
from dotenv import load_dotenv
from pydantic import BaseModel

class AgentResponse(BaseModel):
    answer: str
    success: bool
load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    groq_api_key=os.environ.get("GROQ_API_KEY")
)
tavily = TavilySearch(
    max_result=5,
    tavily_api_key=os.environ.get("TAVILY_API_KEY")
)
@tool
def calculate(a:float,b:float, operation_type:str ):
    """Performs basic arithmetic using two numbers and one operation. Supported operations are
            addition (+), subtraction (-), multiplication (*), and division (/). Division by zero is not allowed.l"""
    operation_type = operation_type.strip().lower()
    if operation_type == "+":
        return a+b
    elif operation_type == "-":
        return a-b
    elif operation_type == "*":
        return a*b
    elif operation_type == "/" and b != 0:
        return a/b
    else:
        raise ValueError(f"Unsupported operation: {operation_type}")
@tool
def get_time():
    """Get the current date and time."""
    return datetime.now().isoformat()
@tool
def web_search(query:str):
    """Search the web for current information."""
    try:
        result = tavily.invoke({
            "query": query
        })
        return str(result)
    except Exception as e:
        raise RuntimeError(
            f"Web search failed: {str(e)}"
        )
def streamIt(response):
    full_response = ""
    for chunk in response:
        print(chunk.content, end="", flush=True)
        full_response += chunk.content
    return full_response
agent = create_agent(
    model=llm,
    tools=[calculate, get_time, web_search],
    system_prompt=("""You are a tutor for school-going kids. Your answers should be short and easy to understand for kids.""")
)
while True:
    user_input = input("Enter your question...").strip()
    if user_input.lower() in ["stop", "exit", "break", "quit"]:
        break
    for chunk in agent.stream(
        {
            "messages": [
                {
                    "role": "user",
                    "content": user_input
                }
            ]
        },
        stream_mode="updates"
    ):
        print(chunk)

AI: 25 × 18 = 450.
